# RunPod

**Upload the Dataset via Google Drive using terminal of Runpod to avoid corruption of file**

1. *On Google Drive*:
   * Find the file
   * Share (read only)
   * Copy link
2. *Open Runpod Terminal and Run*:
   * pip install gdown
   * gdown --fuzzy "https://drive.google.com/file/d/xxxxxxxxxxxxxxxxxxxxxxxx/view?usp=sharing" \
    -O /workspace/MyDataset.csv
3. *Verify upload in Runpod Terminal*:
   * ls -lh /workspace
4. Upload the notebook in Runpod Jupiter Server and run.

------------------------------------------------------
* Shoutout to Hugz
https://www.youtube.com/watch?v=kAL8d6oAGs8

## Install Dependences & Set Device

In [ ]:
%pip install -U \
    sentence-transformers \
    transformers \
    accelerate \
    einops \
    numpy \
    pandas \
    scikit-learn \
    tqdm


In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"


In [ ]:
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


## Load Model

### Qwen3-Embedding-8B

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

model_name = "Qwen/Qwen3-Embedding-8B"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    device_map="cuda"
)


In [ ]:
def last_token_pool(last_hidden_states, attention_mask):
    # Qwen uses "last non-masked token" pooling
    lengths = attention_mask.sum(dim=1) - 1
    return last_hidden_states[torch.arange(last_hidden_states.size(0)), lengths]


In [ ]:
import torch
import torch.nn.functional as F

def embed(text):
    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=32768,
        padding=True
    ).to("cuda")

    with torch.no_grad():
        out = model(**encoded)
        last_hidden = out.last_hidden_state
        attention_mask = encoded["attention_mask"]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        sum_embeddings = torch.sum(last_hidden * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        pooled = sum_embeddings / sum_mask
        pooled = F.normalize(pooled, p=2, dim=1)
        
        return pooled.to(torch.float32).cpu().numpy()

## Paragraphs

### Data import

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm

# 1. Load and prepare the data
df_par = pd.read_csv("01_par_dataset_clean.csv", encoding="utf-8-sig")
print(f"Rows: {len(df_par)}")

# Safety check for the column
text_col = "paragraph"
if text_col not in df_par.columns:
    raise ValueError(f"Column '{text_col}' not found. Available columns: {df_par.columns.tolist()}")

texts_par = df_par[text_col].fillna("").astype(str).tolist()
print(f"Total paragraphs to process: {len(texts_par)}")



### Generate Embeddings

In [ ]:
# 2. The Embedding Loop with Checkpoints
SAVE_EVERY = 10000 
emb_list_par = []

for i, t in enumerate(tqdm(texts_par, desc="Embedding paragraphs")):
    emb = embed(t)
    emb_list_par.append(emb[0])

    # Checkpoint trigger
    if (i + 1) % SAVE_EVERY == 0:
        np.save(
            f"/workspace/partial_par_embeddings_up_to_{i+1}.npy",
            np.vstack(emb_list_par).astype("float32")
        )
        torch.cuda.empty_cache()

# 3. Final Save (If loop completes successfully)
embeddings_par = np.vstack(emb_list_par).astype("float32")
np.save("/workspace/qwen3_8b_par_embeddings.npy", embeddings_par)
print("Saved final paragraph embeddings:", embeddings_par.shape)

# 4. Generate and save the mapping file immediately
mapping_par = df_par.reset_index()[["index", "paragraph_id"]]
mapping_par.columns = ["row_index", "paragraph_id"]
mapping_par.to_csv("/workspace/qwen3_8b_par_index2id.csv", index=False, encoding="utf-8-sig")
print("Saved paragraph mapping file. All done!")

### In case it breakes

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm

# 1. Load the most recent checkpoint
checkpoint_path = "/workspace/partial_par_embeddings_up_to_90000.npy" # <-- UPDATE THIS NUMBER
emb_before = np.load(checkpoint_path)
print("Loaded so far:", emb_before.shape)

# Ensure 'texts_par' and 'df_par' are still in memory (run step 1 of Part 1 if needed)
start_index = emb_before.shape[0]
end_index = len(texts_par)
print(f"Resuming from paragraph {start_index} to {end_index}")

# 2. Resume embedding the rest
emb_list_par_new = []

for i in tqdm(range(start_index, end_index), desc="Resuming paragraph embeddings"):
    t = texts_par[i]
    emb = embed(t)[0]
    emb_list_par_new.append(emb)

    # Free GPU memory periodically
    if (i + 1) % 5000 == 0:
        torch.cuda.empty_cache()

# 3. Stitch the backup and the new data together
emb_after = np.vstack(emb_list_par_new).astype("float32")
emb_all = np.vstack([emb_before, emb_after])

print("Final stitched matrix shape:", emb_all.shape)

# 4. Save the final files
np.save("/workspace/qwen3_8b_par_embeddings.npy", emb_all)

# Generate mapping here too, just in case!
mapping_par = df_par.reset_index()[["index", "paragraph_id"]]
mapping_par.columns = ["row_index", "paragraph_id"]
mapping_par.to_csv("/workspace/qwen3_8b_par_index2id.csv", index=False, encoding="utf-8-sig")

print("Saved final merged embeddings and mapping file. Rescue complete!")

## Narratives

### Import Narratives

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm

# 1. Load and prepare the data
df_narratives = pd.read_csv("05_tempi_df_R1_final_narratives.csv", encoding="utf-8-sig")
print(f"Loaded {len(df_narratives)} rows.")

In [ ]:
text_col = "narrative"
if text_col not in df_narratives.columns:
    raise ValueError(f"Column '{text_col}' not found. Available columns: {df_narratives.columns.tolist()}")

# Clean missing values and convert to a list of strings
texts = df_narratives[text_col].fillna("").astype(str).tolist()
print(f"Total narratives to embed: {len(texts)}")


### Generate embeddings

In [ ]:
from tqdm.auto import tqdm 
import numpy as np
import torch

# 2. Setup the embedding loop with safety checkpoints
SAVE_EVERY = 1000 # Saves a backup to disk every 1,000 narratives
emb_list = []

for i, t in enumerate(tqdm(texts, desc="Embedding narratives")):
    emb = embed(t)
    emb_list.append(emb[0])

    # Checkpoint trigger: Save to disk and clear GPU memory
    if (i + 1) % SAVE_EVERY == 0:
        np.save(
            f"partial_narrative_embeddings_up_to_{i+1}.npy",
            np.vstack(emb_list).astype("float32")
        )
        torch.cuda.empty_cache()

# 3. Final save (If the loop finishes without crashing)
embeddings = np.vstack(emb_list).astype("float32")
np.save("qwen3_8b_narrative_embeddings.npy", embeddings)
print("Saved final embeddings:", embeddings.shape)

# 4. Save the mapping file
mapping = df_narratives.reset_index()[["index"]].rename(columns={"index": "row_index"})
mapping.to_csv("qwen3_8b_narrative_index2row.csv", index=False, encoding="utf-8-sig")
print("Saved mapping file.")

### In case it breaks

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm

# 1. Point to your most recent backup file
checkpoint_path = "partial_narrative_embeddings_up_to_18000.npy" # <-- UPDATE THIS NUMBER
emb_before = np.load(checkpoint_path)
print("Loaded so far:", emb_before.shape)

# Ensure 'texts' is loaded in your environment (run the data prep from Part 1 if needed)
start_index = emb_before.shape[0]
end_index = len(texts)

print(f"Resuming from narrative {start_index} to {end_index}")

# 2. Resume the loop for the remaining narratives
emb_list_new = []

for i in tqdm(range(start_index, end_index), desc="Resuming speeches"):
    t = texts[i]
    emb = embed(t)[0]
    emb_list_new.append(emb)

    # Clear GPU memory periodically (every 500 narratives)
    if (i + 1) % 500 == 0:
        torch.cuda.empty_cache()

# 3. Stitch the old data and the new data together
emb_after = np.vstack(emb_list_new).astype("float32")
emb_all = np.vstack([emb_before, emb_after])

print("Final stitched matrix shape:", emb_all.shape)

# 4. Save the final merged dataset and mapping
np.save("qwen3_8b_narrative_embeddings.npy", emb_all)

mapping = df_narratives.reset_index()[["index"]].rename(columns={"index": "row_index"})
mapping.to_csv("qwen3_8b_narrative_index2row.csv", index=False, encoding="utf-8-sig")
print("Saved final merged embeddings and mapping file. Rescue complete!")